In [ ]:
# CELL 1 - Runtime Information
from pathlib import Path
import platform, sys
from importlib.metadata import version, PackageNotFoundError
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
for package in ["pandas", "numpy", "matplotlib", "scikit-learn", "pytest"]:
    try:
        print(f"{package}={version(package)}")
    except PackageNotFoundError:
        print(f"{package}=NOT INSTALLED")


In [ ]:
# CELL 2 - Install Dependencies and Create Directories
from pathlib import Path
import subprocess, sys
requirements_text = 'pandas>=2.2,<3.0\nnumpy>=1.26,<3.0\nmatplotlib>=3.8,<4.0\nscikit-learn>=1.4,<2.0\npytest>=8.0,<10.0\n'
Path("/content/requirements_stage6.txt").write_text(requirements_text, encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/requirements_stage6.txt"], check=True)
PROJECT_ROOT = Path("/content/DT25_Stage6_RFM_Execution")
PATHS = {
    "src": PROJECT_ROOT / "src", "tests": PROJECT_ROOT / "tests",
    "input": PROJECT_ROOT / "data" / "input", "processed": PROJECT_ROOT / "data" / "processed",
    "tables": PROJECT_ROOT / "outputs" / "tables" / "rfm",
    "figures": PROJECT_ROOT / "outputs" / "figures" / "rfm",
    "logs": PROJECT_ROOT / "outputs" / "logs",
}
for path in PATHS.values(): path.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)


In [ ]:
# CELL 3 - Upload online_retail_rfm_eligible.csv
from google.colab import files
from pathlib import Path
import shutil
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"Upload exactly one file; received {list(uploaded)}")
uploaded_name = next(iter(uploaded))
if uploaded_name != "online_retail_rfm_eligible.csv":
    raise ValueError(f"Required exact file name: online_retail_rfm_eligible.csv; received {uploaded_name}")
uploaded_path = Path("/content") / uploaded_name
if not uploaded_path.exists() or uploaded_path.stat().st_size <= 0:
    raise ValueError("Uploaded input is missing or empty.")
INPUT_PATH = PATHS["input"] / "online_retail_rfm_eligible.csv"
shutil.copy2(uploaded_path, INPUT_PATH)
print("Uploaded physical path:", uploaded_path)
print("Project input path:", INPUT_PATH)
print("Bytes:", INPUT_PATH.stat().st_size)


In [ ]:
# CELL 4 - Validate Input and Locked Baseline
import pandas as pd
EXPECTED_ROWS = 392692
EXPECTED_CUSTOMERS = 4338
REQUIRED_COLUMNS = ["CustomerID", "InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice", "TransactionAmount"]
preview = pd.read_csv(INPUT_PATH, dtype={"CustomerID":"string", "InvoiceNo":"string"}, low_memory=False)
missing = [column for column in REQUIRED_COLUMNS if column not in preview.columns]
if missing: raise ValueError(f"Missing required columns: {missing}")
if len(preview) != EXPECTED_ROWS: raise ValueError(f"Input rows {len(preview)} != accepted baseline {EXPECTED_ROWS}")
if preview["CustomerID"].nunique(dropna=True) != EXPECTED_CUSTOMERS:
    raise ValueError(f"Input customers {preview['CustomerID'].nunique(dropna=True)} != accepted baseline {EXPECTED_CUSTOMERS}")
preview["InvoiceDate"] = pd.to_datetime(preview["InvoiceDate"], errors="coerce")
for column in ["Quantity", "UnitPrice", "TransactionAmount"]: preview[column] = pd.to_numeric(preview[column], errors="coerce")
checks = {
    "CustomerIDMissing": int(preview["CustomerID"].isna().sum()),
    "InvoiceNoMissing": int(preview["InvoiceNo"].isna().sum()),
    "InvalidInvoiceDate": int(preview["InvoiceDate"].isna().sum()),
    "InvalidQuantity": int((preview["Quantity"].isna() | preview["Quantity"].le(0)).sum()),
    "InvalidUnitPrice": int((preview["UnitPrice"].isna() | preview["UnitPrice"].le(0)).sum()),
    "InvalidAmount": int((preview["TransactionAmount"].isna() | preview["TransactionAmount"].le(0)).sum()),
    "CancellationInvoice": int(preview["InvoiceNo"].str.strip().str.upper().str.startswith("C", na=False).sum()),
}
if any(checks.values()): raise ValueError(f"Input eligibility validation failed: {checks}")
print("Input validation: PASS", preview.shape, checks)


In [ ]:
# CELL 5 - Input SHA-256 Baseline
import hashlib, json
from datetime import datetime, timezone
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""): digest.update(block)
    return digest.hexdigest()
INPUT_SHA256_BEFORE = sha256_file(INPUT_PATH)
input_baseline = {
    "Path": "data/input/online_retail_rfm_eligible.csv", "SizeBytes": INPUT_PATH.stat().st_size,
    "Rows": len(preview), "Customers": int(preview["CustomerID"].nunique()),
    "SHA256": INPUT_SHA256_BEFORE, "CapturedUTC": datetime.now(timezone.utc).isoformat(),
}
(PATHS["logs"] / "input_baseline.json").write_text(json.dumps(input_baseline, indent=2), encoding="utf-8")
print(json.dumps(input_baseline, indent=2))


In [ ]:
# CELL 6 - Write and Import rfm_analyzer.py
import importlib, sys
module_source = '"""RFM feature engineering and preprocessing candidates for DT25.\n\nThis module reads an approved RFM-eligible transaction CSV, creates continuous\nRFM features, descriptive quantile scores, customer-level outlier flags, and\nthree clustering-input candidates. It never modifies the input file and does\nnot run clustering.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\nimport hashlib\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import RobustScaler, StandardScaler\n\nREQUIRED_COLUMNS = [\n    "CustomerID", "InvoiceNo", "InvoiceDate", "Quantity",\n    "UnitPrice", "TransactionAmount",\n]\nEXPECTED_INPUT_ROWS = 392_692\nEXPECTED_INPUT_CUSTOMERS = 4_338\nRFM_COLUMNS = ["Recency", "Frequency", "Monetary"]\nRANDOM_STATE = 42\n\n\nclass RFMInputValidationError(ValueError):\n    """Raised when the uploaded processed input violates its accepted baseline."""\n\n\nclass RFMAcceptanceError(RuntimeError):\n    """Raised when an RFM acceptance condition fails."""\n\n\n@dataclass(frozen=True)\nclass ProjectPaths:\n    """Project paths rooted at a caller-provided directory."""\n    root: Path\n\n    @property\n    def input_file(self) -> Path:\n        return self.root / "data" / "input" / "online_retail_rfm_eligible.csv"\n\n    @property\n    def processed_dir(self) -> Path:\n        return self.root / "data" / "processed"\n\n    @property\n    def table_dir(self) -> Path:\n        return self.root / "outputs" / "tables" / "rfm"\n\n    @property\n    def figure_dir(self) -> Path:\n        return self.root / "outputs" / "figures" / "rfm"\n\n    @property\n    def log_dir(self) -> Path:\n        return self.root / "outputs" / "logs"\n\n    def create_output_dirs(self) -> None:\n        for path in (self.processed_dir, self.table_dir, self.figure_dir, self.log_dir):\n            path.mkdir(parents=True, exist_ok=True)\n\n\n@dataclass\nclass RFMArtifacts:\n    """All data artifacts generated before export."""\n    rfm: pd.DataFrame\n    rfm_with_scores: pd.DataFrame\n    standard_raw: pd.DataFrame\n    log_standard: pd.DataFrame\n    robust_raw: pd.DataFrame\n    manual_validation: pd.DataFrame\n    distribution_summary: pd.DataFrame\n    outlier_summary: pd.DataFrame\n    score_distribution: pd.DataFrame\n    preprocessing_comparison: pd.DataFrame\n    acceptance_matrix: pd.DataFrame\n\n\nclass RFMAnalyzer:\n    """Build reproducible RFM features from accepted transaction records."""\n\n    def __init__(self, paths: ProjectPaths, random_state: int = RANDOM_STATE) -> None:\n        self.paths = paths\n        self.random_state = random_state\n        self.transactions: pd.DataFrame | None = None\n        self.reference_date: pd.Timestamp | None = None\n        self.input_sha256_before: str | None = None\n        self.input_sha256_after: str | None = None\n        self.min_invoice_date: pd.Timestamp | None = None\n        self.max_invoice_date: pd.Timestamp | None = None\n\n    @staticmethod\n    def sha256_file(path: Path) -> str:\n        """Return a file\'s SHA-256 hash."""\n        digest = hashlib.sha256()\n        with path.open("rb") as stream:\n            for block in iter(lambda: stream.read(1024 * 1024), b""):\n                digest.update(block)\n        return digest.hexdigest()\n\n    def validate_input(self) -> pd.DataFrame:\n        """Read and validate the accepted RFM-eligible CSV and locked baselines."""\n        path = self.paths.input_file\n        if not path.exists():\n            raise FileNotFoundError(f"Input file not found: {path}")\n        if path.stat().st_size <= 0:\n            raise RFMInputValidationError("Input CSV is empty.")\n        try:\n            df = pd.read_csv(\n                path,\n                dtype={"CustomerID": "string", "InvoiceNo": "string"},\n                low_memory=False,\n            )\n        except (OSError, pd.errors.ParserError, UnicodeDecodeError) as exc:\n            raise RFMInputValidationError(f"Input CSV cannot be read: {exc}") from exc\n        missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]\n        if missing:\n            raise RFMInputValidationError(f"Missing required columns: {missing}")\n\n        df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")\n        for column in ("Quantity", "UnitPrice", "TransactionAmount"):\n            df[column] = pd.to_numeric(df[column], errors="coerce")\n\n        failures: list[str] = []\n        if len(df) != EXPECTED_INPUT_ROWS:\n            failures.append(f"rows={len(df)} expected={EXPECTED_INPUT_ROWS}")\n        customers = int(df["CustomerID"].nunique(dropna=True))\n        if customers != EXPECTED_INPUT_CUSTOMERS:\n            failures.append(f"customers={customers} expected={EXPECTED_INPUT_CUSTOMERS}")\n        if df["CustomerID"].isna().any(): failures.append("CustomerID contains missing values")\n        if df["InvoiceNo"].isna().any(): failures.append("InvoiceNo contains missing values")\n        if df["InvoiceDate"].isna().any(): failures.append("InvoiceDate contains invalid values")\n        if not df["Quantity"].gt(0).all(): failures.append("Quantity contains non-positive or invalid values")\n        if not df["UnitPrice"].gt(0).all(): failures.append("UnitPrice contains non-positive or invalid values")\n        if not df["TransactionAmount"].gt(0).all(): failures.append("TransactionAmount contains non-positive or invalid values")\n        if df["InvoiceNo"].str.strip().str.upper().str.startswith("C", na=False).any():\n            failures.append("Cancellation invoice remains in input")\n        if failures:\n            raise RFMInputValidationError("; ".join(failures))\n        self.transactions = df\n        return df\n\n    def determine_reference_date(self) -> pd.Timestamp:\n        """Set reference date to max valid invoice date plus exactly one day."""\n        if self.transactions is None:\n            raise RFMAcceptanceError("validate_input must run first.")\n        self.min_invoice_date = self.transactions["InvoiceDate"].min()\n        self.max_invoice_date = self.transactions["InvoiceDate"].max()\n        self.reference_date = self.max_invoice_date + pd.Timedelta(days=1)\n        if self.reference_date - self.max_invoice_date != pd.Timedelta(days=1):\n            raise RFMAcceptanceError("Reference date is not exactly max date plus one day.")\n        return self.reference_date\n\n    def compute_rfm(self) -> pd.DataFrame:\n        """Aggregate one RFM row per CustomerID."""\n        if self.transactions is None or self.reference_date is None:\n            raise RFMAcceptanceError("Input validation and reference date are required.")\n        grouped = self.transactions.groupby("CustomerID", as_index=False).agg(\n            LastPurchaseDate=("InvoiceDate", "max"),\n            Frequency=("InvoiceNo", "nunique"),\n            Monetary=("TransactionAmount", "sum"),\n        )\n        grouped["Recency"] = (\n            self.reference_date.normalize() - grouped["LastPurchaseDate"].dt.normalize()\n        ).dt.days.astype("int64")\n        grouped["Frequency"] = grouped["Frequency"].astype("int64")\n        grouped = grouped[["CustomerID", "LastPurchaseDate", "Recency", "Frequency", "Monetary"]]\n        return grouped.sort_values("CustomerID", kind="stable").reset_index(drop=True)\n\n    @staticmethod\n    def validate_rfm(rfm: pd.DataFrame) -> pd.DataFrame:\n        """Return an acceptance matrix and raise for an invalid RFM table."""\n        checks = [\n            ("RFM-01", "One row per accepted customer", len(rfm) == EXPECTED_INPUT_CUSTOMERS),\n            ("RFM-02", "CustomerID is unique", rfm["CustomerID"].is_unique),\n            ("RFM-03", "No missing RFM values", not rfm.isna().any().any()),\n            ("RFM-04", "Recency is integer and non-negative", pd.api.types.is_integer_dtype(rfm["Recency"]) and rfm["Recency"].ge(0).all()),\n            ("RFM-05", "Frequency is integer and positive", pd.api.types.is_integer_dtype(rfm["Frequency"]) and rfm["Frequency"].gt(0).all()),\n            ("RFM-06", "Monetary is positive", rfm["Monetary"].gt(0).all()),\n        ]\n        matrix = pd.DataFrame([\n            {"CheckID": key, "Condition": text, "Result": bool(result), "Status": "PASS" if result else "FAIL"}\n            for key, text, result in checks\n        ])\n        if not matrix["Status"].eq("PASS").all():\n            raise RFMAcceptanceError(f"RFM validation failed:\\n{matrix.loc[matrix.Status.eq(\'FAIL\')]}")\n        return matrix\n\n    def create_manual_validation_sample(self, rfm: pd.DataFrame) -> pd.DataFrame:\n        """Validate five required customer-selection roles against source transactions."""\n        if self.transactions is None or self.reference_date is None:\n            raise RFMAcceptanceError("Transactions and reference date are required.")\n        customer_numeric = pd.to_numeric(rfm["CustomerID"], errors="raise")\n        selectors = [\n            ("MIN_CUSTOMER_ID", rfm.loc[customer_numeric.idxmin(), "CustomerID"]),\n            ("MAX_CUSTOMER_ID", rfm.loc[customer_numeric.idxmax(), "CustomerID"]),\n            ("MAX_FREQUENCY", rfm.loc[rfm["Frequency"].idxmax(), "CustomerID"]),\n            ("MAX_MONETARY", rfm.loc[rfm["Monetary"].idxmax(), "CustomerID"]),\n        ]\n        rng = np.random.default_rng(self.random_state)\n        available = [value for value in rfm["CustomerID"].tolist() if value not in {item[1] for item in selectors}]\n        random_customer = available[int(rng.integers(0, len(available)))]\n        selectors.append((f"RANDOM_STATE_{self.random_state}", random_customer))\n\n        records: list[dict[str, Any]] = []\n        indexed_rfm = rfm.set_index("CustomerID")\n        for role, customer_id in selectors:\n            source = self.transactions.loc[self.transactions["CustomerID"].eq(customer_id)]\n            source_last = source["InvoiceDate"].max()\n            source_frequency = int(source["InvoiceNo"].nunique())\n            source_monetary = float(source["TransactionAmount"].sum())\n            source_recency = int((self.reference_date.normalize() - source_last.normalize()).days)\n            row = indexed_rfm.loc[customer_id]\n            last_match = bool(source_last == row["LastPurchaseDate"])\n            frequency_match = bool(source_frequency == int(row["Frequency"]))\n            monetary_match = bool(np.isclose(source_monetary, float(row["Monetary"]), rtol=1e-9, atol=1e-6))\n            recency_match = bool(source_recency == int(row["Recency"]))\n            records.append({\n                "SelectionRole": role, "CustomerID": customer_id,\n                "SourceLastPurchaseDate": source_last, "RFM_LastPurchaseDate": row["LastPurchaseDate"],\n                "SourceFrequency": source_frequency, "RFM_Frequency": int(row["Frequency"]),\n                "SourceMonetary": source_monetary, "RFM_Monetary": float(row["Monetary"]),\n                "SourceRecency": source_recency, "RFM_Recency": int(row["Recency"]),\n                "LastDateMatch": last_match, "FrequencyMatch": frequency_match,\n                "MonetaryMatch": monetary_match, "RecencyMatch": recency_match,\n                "ValidationStatus": "PASS" if all([last_match, frequency_match, monetary_match, recency_match]) else "FAIL",\n            })\n        result = pd.DataFrame(records)\n        if len(result) != 5 or not result["ValidationStatus"].eq("PASS").all():\n            raise RFMAcceptanceError("Manual five-customer validation failed.")\n        return result\n\n    @staticmethod\n    def compute_distribution_summary(rfm: pd.DataFrame) -> pd.DataFrame:\n        """Compute required distribution statistics for continuous RFM."""\n        records = []\n        for variable in RFM_COLUMNS:\n            series = rfm[variable]\n            records.append({\n                "Variable": variable, "Count": int(series.count()), "Mean": float(series.mean()),\n                "Std": float(series.std()), "Min": float(series.min()), "Q1": float(series.quantile(.25)),\n                "Median": float(series.median()), "Q3": float(series.quantile(.75)), "Max": float(series.max()),\n                "Skewness": float(series.skew()), "Q90": float(series.quantile(.90)),\n                "Q95": float(series.quantile(.95)), "Q99": float(series.quantile(.99)),\n                "Q999": float(series.quantile(.999)), "MissingCount": int(series.isna().sum()),\n                "InvalidCount": int((series.lt(0) if variable == "Recency" else series.le(0)).sum()),\n            })\n        return pd.DataFrame(records)\n\n    @staticmethod\n    def flag_customer_outliers(rfm: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n        """Flag customer RFM outliers with IQR and retain every customer."""\n        out = rfm.copy()\n        records = []\n        for variable in RFM_COLUMNS:\n            q1, q3 = out[variable].quantile([.25, .75])\n            iqr = q3 - q1\n            lower, upper = float(q1 - 1.5 * iqr), float(q3 + 1.5 * iqr)\n            flag_name = f"{variable}_Outlier"\n            out[flag_name] = out[variable].lt(lower) | out[variable].gt(upper)\n            records.append({\n                "Variable": variable, "Method": "IQR 1.5", "Q1": float(q1), "Q3": float(q3),\n                "IQR": float(iqr), "LowerLimit": lower, "UpperLimit": upper,\n                "FlaggedCustomers": int(out[flag_name].sum()),\n                "FlaggedRatePercent": float(out[flag_name].mean() * 100),\n                "Treatment": "FLAG_ONLY_NO_AUTOMATIC_REMOVAL",\n            })\n        flag_columns = [f"{variable}_Outlier" for variable in RFM_COLUMNS]\n        out["Any_RFM_Outlier"] = out[flag_columns].any(axis=1)\n        records.append({\n            "Variable": "Any_RFM_Outlier", "Method": "Union of three IQR flags",\n            "Q1": np.nan, "Q3": np.nan, "IQR": np.nan, "LowerLimit": np.nan, "UpperLimit": np.nan,\n            "FlaggedCustomers": int(out["Any_RFM_Outlier"].sum()),\n            "FlaggedRatePercent": float(out["Any_RFM_Outlier"].mean() * 100),\n            "Treatment": "FLAG_ONLY_REVIEW_CONTEXT",\n        })\n        return out, pd.DataFrame(records)\n\n    @staticmethod\n    def _rank_quantile_score(series: pd.Series, reverse: bool = False) -> pd.Series:\n        """Create stable quintile scores despite duplicated raw values."""\n        percentile = series.rank(method="first", pct=True)\n        score = np.ceil(percentile * 5).clip(1, 5).astype("int64")\n        return (6 - score).astype("int64") if reverse else score\n\n    def create_rfm_scores(self, rfm_flagged: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n        """Create descriptive 1-5 scores without replacing continuous clustering features."""\n        out = rfm_flagged.copy()\n        out["R_Score"] = self._rank_quantile_score(out["Recency"], reverse=True)\n        out["F_Score"] = self._rank_quantile_score(out["Frequency"], reverse=False)\n        out["M_Score"] = self._rank_quantile_score(out["Monetary"], reverse=False)\n        out["RFM_Score_String"] = out[["R_Score", "F_Score", "M_Score"]].astype(str).agg("".join, axis=1)\n        out["RFM_TotalScore"] = out[["R_Score", "F_Score", "M_Score"]].sum(axis=1).astype("int64")\n        for column in ("R_Score", "F_Score", "M_Score"):\n            if not out[column].between(1, 5).all():\n                raise RFMAcceptanceError(f"{column} contains a value outside 1..5")\n        distribution = (\n            out.groupby(["R_Score", "F_Score", "M_Score"], as_index=False)\n            .size().rename(columns={"size": "Customers"})\n        )\n        return out, distribution\n\n    @staticmethod\n    def build_preprocessing_candidates(rfm: pd.DataFrame) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:\n        """Build three verified feature candidates without selecting a winner."""\n        features = rfm[RFM_COLUMNS].astype("float64")\n        transformed = {\n            "STANDARD_RAW": StandardScaler().fit_transform(features),\n            "LOG_STANDARD": StandardScaler().fit_transform(np.log1p(features)),\n            "ROBUST_RAW": RobustScaler().fit_transform(features),\n        }\n        candidates: dict[str, pd.DataFrame] = {}\n        comparisons: list[dict[str, Any]] = []\n        for name, matrix in transformed.items():\n            frame = pd.DataFrame(matrix, columns=[f"{column}_Scaled" for column in RFM_COLUMNS])\n            frame.insert(0, "CustomerID", rfm["CustomerID"].to_numpy())\n            numeric = frame.drop(columns="CustomerID")\n            same_ids = frame["CustomerID"].astype("string").reset_index(drop=True).equals(\n                rfm["CustomerID"].astype("string").reset_index(drop=True)\n            )\n            if len(frame) != len(rfm) or not same_ids:\n                raise RFMAcceptanceError(f"Customer identity changed in candidate {name}")\n            if numeric.isna().any().any() or not np.isfinite(numeric.to_numpy()).all():\n                raise RFMAcceptanceError(f"NaN or infinity in candidate {name}")\n            candidates[name] = frame\n            for column in numeric.columns:\n                comparisons.append({\n                    "Candidate": name, "Feature": column,\n                    "Rows": len(frame), "Customers": int(frame["CustomerID"].nunique()),\n                    "MissingCount": int(numeric[column].isna().sum()),\n                    "InfiniteCount": int(np.isinf(numeric[column]).sum()),\n                    "Mean": float(numeric[column].mean()), "Median": float(numeric[column].median()),\n                    "Std": float(numeric[column].std(ddof=0)), "Min": float(numeric[column].min()),\n                    "Max": float(numeric[column].max()), "Skewness": float(numeric[column].skew()),\n                    "SelectionStatus": "CANDIDATE_NOT_SELECTED",\n                })\n        return candidates, pd.DataFrame(comparisons)\n\n    def create_evidence_summary(self, artifacts: RFMArtifacts) -> pd.DataFrame:\n        """Create a single-row execution summary from computed artifacts."""\n        if self.transactions is None or self.reference_date is None:\n            raise RFMAcceptanceError("Execution state incomplete.")\n        rfm = artifacts.rfm\n        return pd.DataFrame([{\n            "INPUT_ROWS": len(self.transactions),\n            "INPUT_CUSTOMERS": int(self.transactions["CustomerID"].nunique()),\n            "MIN_INVOICE_DATE": self.min_invoice_date,\n            "MAX_INVOICE_DATE": self.max_invoice_date,\n            "REFERENCE_DATE": self.reference_date,\n            "RFM_ROWS": len(rfm), "RFM_CUSTOMERS": int(rfm["CustomerID"].nunique()),\n            "RFM_DUPLICATE_CUSTOMER_IDS": int(rfm["CustomerID"].duplicated().sum()),\n            "RFM_MISSING_VALUES": int(rfm.isna().sum().sum()),\n            "RFM_INVALID_RECENCY": int(rfm["Recency"].lt(0).sum()),\n            "RFM_INVALID_FREQUENCY": int(rfm["Frequency"].le(0).sum()),\n            "RFM_INVALID_MONETARY": int(rfm["Monetary"].le(0).sum()),\n            "MANUAL_VALIDATION_PASS": int(artifacts.manual_validation["ValidationStatus"].eq("PASS").sum()),\n            "RFM_OUTLIER_CUSTOMERS": int(artifacts.rfm_with_scores["Any_RFM_Outlier"].sum()),\n        }])\n\n    def export_outputs(self, artifacts: RFMArtifacts) -> dict[str, Path]:\n        """Export ten required CSV files; figures are created by the notebook."""\n        self.paths.create_output_dirs()\n        outputs = {\n            "rfm_customers": self.paths.processed_dir / "rfm_customers.csv",\n            "rfm_with_scores": self.paths.processed_dir / "rfm_with_scores.csv",\n            "rfm_standard_raw": self.paths.processed_dir / "rfm_standard_raw.csv",\n            "rfm_log_standard": self.paths.processed_dir / "rfm_log_standard.csv",\n            "rfm_robust_raw": self.paths.processed_dir / "rfm_robust_raw.csv",\n            "rfm_distribution_summary": self.paths.table_dir / "rfm_distribution_summary.csv",\n            "rfm_manual_validation": self.paths.table_dir / "rfm_manual_validation.csv",\n            "rfm_outlier_summary": self.paths.table_dir / "rfm_outlier_summary.csv",\n            "rfm_score_distribution": self.paths.table_dir / "rfm_score_distribution.csv",\n            "rfm_preprocessing_comparison": self.paths.table_dir / "rfm_preprocessing_comparison.csv",\n            "rfm_acceptance_matrix": self.paths.table_dir / "rfm_acceptance_matrix.csv",\n        }\n        frames = {\n            "rfm_customers": artifacts.rfm,\n            "rfm_with_scores": artifacts.rfm_with_scores,\n            "rfm_standard_raw": artifacts.standard_raw,\n            "rfm_log_standard": artifacts.log_standard,\n            "rfm_robust_raw": artifacts.robust_raw,\n            "rfm_distribution_summary": artifacts.distribution_summary,\n            "rfm_manual_validation": artifacts.manual_validation,\n            "rfm_outlier_summary": artifacts.outlier_summary,\n            "rfm_score_distribution": artifacts.score_distribution,\n            "rfm_preprocessing_comparison": artifacts.preprocessing_comparison,\n            "rfm_acceptance_matrix": artifacts.acceptance_matrix,\n        }\n        for key, path in outputs.items():\n            frames[key].to_csv(path, index=False, encoding="utf-8-sig")\n        return outputs\n\n    def run(self) -> tuple[RFMArtifacts, dict[str, Path], pd.DataFrame]:\n        """Execute feature engineering without clustering or customer removal."""\n        self.paths.create_output_dirs()\n        self.input_sha256_before = self.sha256_file(self.paths.input_file)\n        self.validate_input()\n        self.determine_reference_date()\n        rfm = self.compute_rfm()\n        acceptance = self.validate_rfm(rfm)\n        manual = self.create_manual_validation_sample(rfm)\n        distribution = self.compute_distribution_summary(rfm)\n        flagged, outlier_summary = self.flag_customer_outliers(rfm)\n        scored, score_distribution = self.create_rfm_scores(flagged)\n        candidates, comparison = self.build_preprocessing_candidates(rfm)\n        artifacts = RFMArtifacts(\n            rfm=rfm, rfm_with_scores=scored,\n            standard_raw=candidates["STANDARD_RAW"],\n            log_standard=candidates["LOG_STANDARD"],\n            robust_raw=candidates["ROBUST_RAW"],\n            manual_validation=manual, distribution_summary=distribution,\n            outlier_summary=outlier_summary, score_distribution=score_distribution,\n            preprocessing_comparison=comparison, acceptance_matrix=acceptance,\n        )\n        outputs = self.export_outputs(artifacts)\n        self.input_sha256_after = self.sha256_file(self.paths.input_file)\n        if self.input_sha256_before != self.input_sha256_after:\n            raise RFMAcceptanceError("Input file checksum changed during execution.")\n        return artifacts, outputs, self.create_evidence_summary(artifacts)\n'
module_path = PATHS["src"] / "rfm_analyzer.py"
module_path.write_text(module_source, encoding="utf-8")
if str(PATHS["src"]) not in sys.path: sys.path.insert(0, str(PATHS["src"]))
rfm_analyzer = importlib.import_module("rfm_analyzer")
importlib.reload(rfm_analyzer)
print("Imported:", module_path)


In [ ]:
# CELL 7 - Compute Continuous RFM
import traceback
from datetime import datetime, timezone
from rfm_analyzer import ProjectPaths, RFMAnalyzer
execution_started = datetime.now(timezone.utc)
try:
    analyzer = RFMAnalyzer(ProjectPaths(PROJECT_ROOT), random_state=42)
    artifacts, csv_output_paths, evidence_summary = analyzer.run()
except Exception:
    traceback_text = traceback.format_exc()
    (PATHS["logs"] / "execution_log.txt").write_text(
        "ENTRY_POINT=RFMAnalyzer(ProjectPaths(PROJECT_ROOT), random_state=42).run()\nSTATUS=FAIL\n" + traceback_text,
        encoding="utf-8")
    raise
print("MIN_INVOICE_DATE =", analyzer.min_invoice_date)
print("MAX_INVOICE_DATE =", analyzer.max_invoice_date)
print("REFERENCE_DATE =", analyzer.reference_date)
print(artifacts.rfm.head())


In [ ]:
# CELL 8 - Validate RFM
acceptance = artifacts.acceptance_matrix.copy()
reference_rule_pass = analyzer.reference_date - analyzer.max_invoice_date == pd.Timedelta(days=1)
acceptance = pd.concat([acceptance, pd.DataFrame([{
    "CheckID":"RFM-07", "Condition":"Reference date equals max InvoiceDate plus one day",
    "Result":bool(reference_rule_pass), "Status":"PASS" if reference_rule_pass else "FAIL"
}])], ignore_index=True)
acceptance.to_csv(PATHS["tables"] / "rfm_acceptance_matrix.csv", index=False, encoding="utf-8-sig")
display(acceptance)
if not acceptance["Status"].eq("PASS").all(): raise AssertionError("RFM acceptance matrix failed.")


In [ ]:
# CELL 9 - Manual Five-Customer Validation
manual_validation = artifacts.manual_validation
manual_validation.to_csv(PATHS["tables"] / "rfm_manual_validation.csv", index=False, encoding="utf-8-sig")
display(manual_validation)
if len(manual_validation) != 5 or not manual_validation["ValidationStatus"].eq("PASS").all():
    raise AssertionError("Manual validation is not 5/5 PASS.")


In [ ]:
# CELL 10 - RFM Distribution Audit and Figures
import matplotlib.pyplot as plt
import numpy as np
artifacts.distribution_summary.to_csv(PATHS["tables"] / "rfm_distribution_summary.csv", index=False, encoding="utf-8-sig")
display(artifacts.distribution_summary)
rfm = artifacts.rfm
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, variable in zip(axes, ["Recency", "Frequency", "Monetary"]):
    ax.hist(rfm[variable], bins=50); ax.set_title(f"Phân bố {variable} gốc"); ax.set_xlabel(variable); ax.set_ylabel("Số khách hàng")
fig.tight_layout(); fig.savefig(PATHS["figures"] / "rfm_histograms_raw.png", dpi=160); plt.close(fig)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, variable in zip(axes, ["Recency", "Frequency", "Monetary"]):
    ax.boxplot(rfm[variable], vert=True); ax.set_title(f"Boxplot {variable} gốc"); ax.set_ylabel(variable)
fig.tight_layout(); fig.savefig(PATHS["figures"] / "rfm_boxplots_raw.png", dpi=160); plt.close(fig)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, variable in zip(axes, ["Recency", "Frequency", "Monetary"]):
    values = np.log1p(rfm[variable]); ax.hist(values, bins=50); ax.set_title(f"Phân bố log1p({variable})"); ax.set_xlabel(f"log1p({variable})"); ax.set_ylabel("Số khách hàng")
fig.tight_layout(); fig.savefig(PATHS["figures"] / "rfm_histograms_log.png", dpi=160); plt.close(fig)
print("Three figures created.")


In [ ]:
# CELL 11 - Customer-Level Outlier Audit
artifacts.outlier_summary.to_csv(PATHS["tables"] / "rfm_outlier_summary.csv", index=False, encoding="utf-8-sig")
display(artifacts.outlier_summary)
print("No customer was removed. Outliers are flags for contextual review only.")


In [ ]:
# CELL 12 - Create Descriptive RFM Scores
score_columns = ["R_Score", "F_Score", "M_Score"]
if not artifacts.rfm_with_scores[score_columns].apply(lambda s: s.between(1,5).all()).all():
    raise AssertionError("RFM score range failed.")
artifacts.score_distribution.to_csv(PATHS["tables"] / "rfm_score_distribution.csv", index=False, encoding="utf-8-sig")
display(artifacts.rfm_with_scores.head())
print("Scores are descriptive only; clustering retains three continuous RFM features.")


In [ ]:
# CELL 13 - Create Three Preprocessing Candidates
artifacts.preprocessing_comparison.to_csv(PATHS["tables"] / "rfm_preprocessing_comparison.csv", index=False, encoding="utf-8-sig")
display(artifacts.preprocessing_comparison)
print("Candidates: STANDARD_RAW, LOG_STANDARD, ROBUST_RAW")
print("No candidate selected and no K-Means executed.")


In [ ]:
# CELL 14 - Export Required Outputs
# Analyzer.run() exported the eleven CSV outputs. Cell 10 exported three required figures.
required_csv_paths = list(csv_output_paths.values())
required_figure_paths = [
    PATHS["figures"] / "rfm_histograms_raw.png",
    PATHS["figures"] / "rfm_boxplots_raw.png",
    PATHS["figures"] / "rfm_histograms_log.png",
]
for path in required_csv_paths + required_figure_paths:
    print(path.relative_to(PROJECT_ROOT), path.stat().st_size if path.exists() else 0)


In [ ]:
# CELL 15 - Read-Back Verification
verification = []
for index, path in enumerate(required_csv_paths, 1):
    record = {"FileID":f"CSV-{index:02d}", "Path":str(path.relative_to(PROJECT_ROOT)), "Exists":path.exists(), "SizeBytes":path.stat().st_size if path.exists() else 0}
    try:
        frame = pd.read_csv(path, low_memory=False)
        if path.stat().st_size <= 0: raise ValueError("empty file")
        record.update({"Readable":True,"Rows":len(frame),"Columns":frame.shape[1],"Status":"PASS","Error":""})
    except (OSError, ValueError, pd.errors.ParserError) as exc:
        record.update({"Readable":False,"Rows":None,"Columns":None,"Status":"FAIL","Error":str(exc)})
    verification.append(record)
for index, path in enumerate(required_figure_paths, 1):
    ok = path.exists() and path.stat().st_size > 0
    verification.append({"FileID":f"FIG-{index:02d}","Path":str(path.relative_to(PROJECT_ROOT)),"Exists":path.exists(),"SizeBytes":path.stat().st_size if path.exists() else 0,"Readable":ok,"Rows":None,"Columns":None,"Status":"PASS" if ok else "FAIL","Error":"" if ok else "missing or empty"})
output_verification = pd.DataFrame(verification)
output_verification.to_csv(PATHS["tables"] / "output_verification.csv", index=False, encoding="utf-8-sig")
display(output_verification)
if not output_verification["Status"].eq("PASS").all(): raise AssertionError("Output verification failed.")


In [ ]:
# CELL 16 - Automated Acceptance Tests
import os, subprocess
test_source = '"""Automated acceptance tests for externally executed Stage 6 RFM outputs."""\nfrom pathlib import Path\nimport hashlib\nimport numpy as np\nimport pandas as pd\n\nROOT = Path(__file__).resolve().parents[1]\nINPUT = ROOT / "data" / "input" / "online_retail_rfm_eligible.csv"\nPROCESSED = ROOT / "data" / "processed"\nTABLES = ROOT / "outputs" / "tables" / "rfm"\nFIGURES = ROOT / "outputs" / "figures" / "rfm"\nEXPECTED_ROWS = 392_692\nEXPECTED_CUSTOMERS = 4_338\n\nCSV_OUTPUTS = [\n    PROCESSED / "rfm_customers.csv",\n    PROCESSED / "rfm_with_scores.csv",\n    PROCESSED / "rfm_standard_raw.csv",\n    PROCESSED / "rfm_log_standard.csv",\n    PROCESSED / "rfm_robust_raw.csv",\n    TABLES / "rfm_distribution_summary.csv",\n    TABLES / "rfm_manual_validation.csv",\n    TABLES / "rfm_outlier_summary.csv",\n    TABLES / "rfm_score_distribution.csv",\n    TABLES / "rfm_preprocessing_comparison.csv",\n    TABLES / "rfm_acceptance_matrix.csv",\n]\nFIGURE_OUTPUTS = [\n    FIGURES / "rfm_histograms_raw.png",\n    FIGURES / "rfm_boxplots_raw.png",\n    FIGURES / "rfm_histograms_log.png",\n]\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef test_input_baseline_and_integrity() -> None:\n    assert INPUT.exists() and INPUT.stat().st_size > 0\n    source = pd.read_csv(INPUT, dtype={"CustomerID": "string", "InvoiceNo": "string"}, low_memory=False)\n    assert len(source) == EXPECTED_ROWS\n    assert source["CustomerID"].nunique() == EXPECTED_CUSTOMERS\n    checksum = pd.read_csv(TABLES / "input_checksum_report.csv")\n    assert checksum.loc[0, "INPUT_SHA256_BEFORE"] == checksum.loc[0, "INPUT_SHA256_AFTER"]\n    assert checksum.loc[0, "INPUT_SHA256_AFTER"] == sha256_file(INPUT)\n\n\ndef test_rfm_table_validity() -> None:\n    rfm = pd.read_csv(PROCESSED / "rfm_customers.csv", dtype={"CustomerID": "string"}, parse_dates=["LastPurchaseDate"])\n    assert len(rfm) == EXPECTED_CUSTOMERS\n    assert rfm["CustomerID"].is_unique\n    assert not rfm.isna().any().any()\n    assert rfm["Recency"].ge(0).all()\n    assert rfm["Frequency"].gt(0).all()\n    assert rfm["Monetary"].gt(0).all()\n\n\ndef test_frequency_matches_source_nunique() -> None:\n    source = pd.read_csv(INPUT, dtype={"CustomerID": "string", "InvoiceNo": "string"}, usecols=["CustomerID", "InvoiceNo"], low_memory=False)\n    expected = source.groupby("CustomerID")["InvoiceNo"].nunique().sort_index()\n    actual = pd.read_csv(PROCESSED / "rfm_customers.csv", dtype={"CustomerID": "string"}).set_index("CustomerID")["Frequency"].sort_index()\n    pd.testing.assert_series_equal(actual, expected, check_names=False, check_dtype=False)\n\n\ndef test_manual_validation_five_of_five() -> None:\n    validation = pd.read_csv(TABLES / "rfm_manual_validation.csv")\n    assert len(validation) == 5\n    assert validation["ValidationStatus"].eq("PASS").all()\n\n\ndef test_score_range() -> None:\n    scored = pd.read_csv(PROCESSED / "rfm_with_scores.csv")\n    for column in ["R_Score", "F_Score", "M_Score"]:\n        assert scored[column].between(1, 5).all()\n\n\ndef test_three_candidates_are_finite_and_aligned() -> None:\n    expected_ids = pd.read_csv(PROCESSED / "rfm_customers.csv", dtype={"CustomerID": "string"})["CustomerID"]\n    for name in ["rfm_standard_raw.csv", "rfm_log_standard.csv", "rfm_robust_raw.csv"]:\n        candidate = pd.read_csv(PROCESSED / name, dtype={"CustomerID": "string"})\n        assert len(candidate) == EXPECTED_CUSTOMERS\n        assert candidate["CustomerID"].equals(expected_ids)\n        numeric = candidate.drop(columns="CustomerID")\n        assert not numeric.isna().any().any()\n        assert np.isfinite(numeric.to_numpy()).all()\n\n\ndef test_required_outputs_and_figures() -> None:\n    for path in CSV_OUTPUTS:\n        assert path.exists() and path.stat().st_size > 0, path\n        pd.read_csv(path, low_memory=False)\n    for path in FIGURE_OUTPUTS:\n        assert path.exists() and path.stat().st_size > 0, path\n\n\ndef test_acceptance_matrix_passes() -> None:\n    matrix = pd.read_csv(TABLES / "rfm_acceptance_matrix.csv")\n    assert matrix["Status"].eq("PASS").all()\n'
test_path = PATHS["tests"] / "test_stage6_acceptance.py"
test_path.write_text(test_source, encoding="utf-8")
# Checksum report must exist before pytest.
INPUT_SHA256_AFTER = sha256_file(INPUT_PATH)
checksum_frame = pd.DataFrame([{
    "INPUT_FILE_PATH":"data/input/online_retail_rfm_eligible.csv",
    "INPUT_SHA256_BEFORE":INPUT_SHA256_BEFORE,"INPUT_SHA256_AFTER":INPUT_SHA256_AFTER,
    "INPUT_FILE_UNCHANGED":INPUT_SHA256_BEFORE == INPUT_SHA256_AFTER,
    "INPUT_SIZE_BEFORE":input_baseline["SizeBytes"],"INPUT_SIZE_AFTER":INPUT_PATH.stat().st_size,
}])
checksum_frame.to_csv(PATHS["tables"] / "input_checksum_report.csv", index=False, encoding="utf-8-sig")
env=os.environ.copy(); env["PYTHONPATH"]=str(PATHS["src"])
pytest_result=subprocess.run([sys.executable,"-m","pytest","-q",str(test_path)],cwd=PROJECT_ROOT,env=env,text=True,capture_output=True)
pytest_text=pytest_result.stdout+"\n"+pytest_result.stderr
(PATHS["logs"] / "pytest_output.txt").write_text(pytest_text,encoding="utf-8")
print(pytest_text)
if pytest_result.returncode != 0: raise AssertionError(f"Pytest failed: {pytest_result.returncode}")


In [ ]:
# CELL 17 - Input SHA-256 After Execution
INPUT_SHA256_AFTER = sha256_file(INPUT_PATH)
INPUT_FILE_UNCHANGED = INPUT_SHA256_BEFORE == INPUT_SHA256_AFTER
checksum_frame = pd.DataFrame([{
    "INPUT_FILE_PATH":"data/input/online_retail_rfm_eligible.csv",
    "INPUT_SHA256_BEFORE":INPUT_SHA256_BEFORE,"INPUT_SHA256_AFTER":INPUT_SHA256_AFTER,
    "INPUT_FILE_UNCHANGED":INPUT_FILE_UNCHANGED,
    "INPUT_SIZE_BEFORE":input_baseline["SizeBytes"],"INPUT_SIZE_AFTER":INPUT_PATH.stat().st_size,
}])
checksum_frame.to_csv(PATHS["tables"] / "input_checksum_report.csv", index=False, encoding="utf-8-sig")
(PATHS["logs"] / "input_checksum_report.txt").write_text(checksum_frame.to_string(index=False), encoding="utf-8")
display(checksum_frame)
if not INPUT_FILE_UNCHANGED: raise AssertionError("CRITICAL FAIL: input CSV changed during execution.")


In [ ]:
# CELL 18 - Final Machine-Readable Acceptance Summary
import platform
from importlib.metadata import version
summary=evidence_summary.iloc[0]
csv_verified=int(output_verification.loc[output_verification.FileID.str.startswith("CSV"),"Status"].eq("PASS").sum())
fig_verified=int(output_verification.loc[output_verification.FileID.str.startswith("FIG"),"Status"].eq("PASS").sum())
failed=[]
if not INPUT_FILE_UNCHANGED: failed.append("INPUT_INTEGRITY")
if csv_verified != len(required_csv_paths): failed.append("CSV_OUTPUTS")
if fig_verified != 3: failed.append("FIGURES")
if pytest_result.returncode != 0: failed.append("PYTEST")
if not acceptance["Status"].eq("PASS").all(): failed.append("RFM_ACCEPTANCE")
if not manual_validation.ValidationStatus.eq("PASS").all(): failed.append("MANUAL_VALIDATION")
candidate="PASS" if not failed else "FAIL"
lines=[
 f"STAGE_6_EXECUTION_STATUS = {candidate}",f"STAGE_6_ACCEPTANCE_CANDIDATE = {candidate}","",
 f"INPUT_ROWS = {int(summary.INPUT_ROWS)}",f"INPUT_CUSTOMERS = {int(summary.INPUT_CUSTOMERS)}",
 f"INPUT_SHA256_BEFORE = {INPUT_SHA256_BEFORE}",f"INPUT_SHA256_AFTER = {INPUT_SHA256_AFTER}",
 f"INPUT_FILE_UNCHANGED = {'YES' if INPUT_FILE_UNCHANGED else 'NO'}","",
 f"REFERENCE_DATE = {pd.Timestamp(summary.REFERENCE_DATE)}",f"RFM_ROWS = {int(summary.RFM_ROWS)}",
 f"RFM_CUSTOMERS = {int(summary.RFM_CUSTOMERS)}",f"RFM_DUPLICATE_CUSTOMER_IDS = {int(summary.RFM_DUPLICATE_CUSTOMER_IDS)}",
 f"RFM_MISSING_VALUES = {int(summary.RFM_MISSING_VALUES)}",f"RFM_INVALID_RECENCY = {int(summary.RFM_INVALID_RECENCY)}",
 f"RFM_INVALID_FREQUENCY = {int(summary.RFM_INVALID_FREQUENCY)}",f"RFM_INVALID_MONETARY = {int(summary.RFM_INVALID_MONETARY)}","",
 f"MANUAL_VALIDATION_PASS = {int(summary.MANUAL_VALIDATION_PASS)}/5",
 f"RFM_SCORE_RANGE_STATUS = {'PASS' if artifacts.rfm_with_scores[['R_Score','F_Score','M_Score']].apply(lambda s:s.between(1,5).all()).all() else 'FAIL'}",
 "PREPROCESSING_CANDIDATES = 3/3 VERIFIED",
 f"REQUIRED_OUTPUT_FILES = {csv_verified}/{len(required_csv_paths)}",
 f"REQUIRED_FIGURES = {fig_verified}/3 VERIFIED",
 f"OUTPUT_READ_BACK = {'PASS' if csv_verified == len(required_csv_paths) else 'FAIL'}",
 f"PYTEST_STATUS = {'PASS' if pytest_result.returncode == 0 else 'FAIL'}",
 f"RFM_TABLE_READINESS = {'READY' if candidate == 'PASS' else 'NOT READY'}",
 f"CLUSTERING_INPUT_READINESS = {'READY' if candidate == 'PASS' else 'NOT READY'}","",
 'OPEN_CONDITIONS = ["Final Chat evidence review pending", "SQLite second-format confirmation remains open"]',
 f"FAILED_CHECKS = {failed if failed else 'NONE'}",
 "NOT_VERIFIED_ITEMS = NONE" if candidate=="PASS" else 'NOT_VERIFIED_ITEMS = ["See failed checks and logs"]',
]
acceptance_summary="\n".join(lines)
(PATHS["logs"] / "acceptance_summary.txt").write_text(acceptance_summary,encoding="utf-8")
execution_finished=datetime.now(timezone.utc)
(PATHS["logs"] / "execution_log.txt").write_text(
    "ENTRY_POINT=RFMAnalyzer(ProjectPaths(PROJECT_ROOT), random_state=42).run()\n"
    f"INPUT={INPUT_PATH}\nSTATUS={candidate}\nSTARTED_UTC={execution_started.isoformat()}\nFINISHED_UTC={execution_finished.isoformat()}\nEXCEPTION=NONE\n",
    encoding="utf-8")
environment="\n".join([f"Python={sys.version}",f"Platform={platform.platform()}",f"pandas={version('pandas')}",f"numpy={version('numpy')}",f"matplotlib={version('matplotlib')}",f"scikit-learn={version('scikit-learn')}",f"pytest={version('pytest')}",f"ProjectRoot={PROJECT_ROOT}"])
(PATHS["logs"] / "environment_info.txt").write_text(environment,encoding="utf-8")
print(acceptance_summary)
if candidate != "PASS": raise AssertionError("Stage 6 acceptance candidate failed.")


In [ ]:
# CELL 19 - Build and Download Evidence ZIP
import zipfile
from google.colab import files
notebook_candidate=Path("/content/DT25_Stage6_RFM_Execution.ipynb")
if not notebook_candidate.exists():
    (PATHS["logs"] / "notebook_provenance.txt").write_text("Notebook executed in Google Colab; download notebook separately if required.",encoding="utf-8")
evidence=[
    PATHS["logs"] / "acceptance_summary.txt", PATHS["logs"] / "execution_log.txt",
    PATHS["logs"] / "environment_info.txt", PATHS["logs"] / "input_checksum_report.txt",
    PATHS["logs"] / "pytest_output.txt", PATHS["tables"] / "output_verification.csv",
    PATHS["tables"] / "input_checksum_report.csv", PATHS["src"] / "rfm_analyzer.py",
    PATHS["tests"] / "test_stage6_acceptance.py", *required_csv_paths, *required_figure_paths,
]
provenance=PATHS["logs"] / "notebook_provenance.txt"
if provenance.exists(): evidence.append(provenance)
if notebook_candidate.exists(): evidence.append(notebook_candidate)
for path in evidence:
    if not path.exists() or path.stat().st_size <= 0: raise FileNotFoundError(f"Evidence missing or empty: {path}")
zip_path=PROJECT_ROOT / "DT25_Stage6_RFM_Evidence.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in evidence:
        arcname=path.relative_to(PROJECT_ROOT) if PROJECT_ROOT in path.parents else Path(path.name)
        archive.write(path,arcname)
print("Evidence ZIP:",zip_path,"bytes=",zip_path.stat().st_size)
files.download(str(zip_path))
